# RE_104 · Quant costa llogar un pis

**Optativa d'Aprenentatge automàtic · Bloc 1 — Dades · Dificultat 2**

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Ets periodista de dades en un diari digital del Vallès. Prepares un article sobre el preu del
lloguer i tens les dades oficials de les fiances de tots els contractes de lloguer de Catalunya des
del 2007.

**La pregunta:** quant ha pujat el lloguer a Barcelona, Sant Cugat del Vallès i Terrassa en deu
anys? On és més car llogar avui? I quants contractes se signen cada any?

L'enunciat complet, amb el diccionari de columnes, és a [RE_104](../RE_104_lloguer.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FITXER = "lloguer_municipis.csv"
URL = "https://raw.githubusercontent.com/ZenidX/Machine-Learning/main/Machine%20Learning/04_reptes/dades/" + FITXER
ruta = URL
for candidata in (os.path.join("dades", FITXER), os.path.join("..", "dades", FITXER)):
    if os.path.exists(candidata):
        ruta = candidata
df = pd.read_csv(ruta)
print(ruta, df.shape)

## Nivell 1 · Explorar

### 1.1 · Forma i municipis

Quantes files i columnes hi ha, i de quins anys? Quants municipis diferents hi ha? Compta'ls de dues maneres: pel nom (`nom_territori`) i pel codi (`codi_territorial`). Si no surt el mateix, busca'n la causa.

In [ ]:
print(df.shape, df["any"].min(), df["any"].max())
print(df["nom_territori"].nunique(), df["codi_territorial"].nunique())
noms = df.groupby("codi_territorial")["nom_territori"].unique()
noms[noms.str.len() > 1]

**Comentari.** 954 noms per a 945 codis: nou municipis han canviat de nom (fusions, ortografia). Agrupar pel nom els parteix en dos.

### 1.2 · Files sense renda

Quantes files no tenen `renda`? Compara la columna `habitatges` de les files amb renda i de les que no en tenen. Què hi veus?

In [ ]:
print(df["renda"].isna().sum(), df["renda"].notna().sum(), len(df))
df.groupby(df["renda"].isna())["habitatges"].describe()

**Comentari.** Les files sense renda tenen sempre 5 contractes o menys: és secret estadístic, no un error.

### 1.3 · Anys i períodes

Quins valors té la columna `periode`? Fes una taula amb els anys a les files, els períodes a les columnes i el nombre de files a cada casella.

In [ ]:
print(df["periode"].value_counts())
taula = df.groupby(["any", "periode"]).size().unstack(fill_value=0)
print(taula.to_numpy().sum(), len(df))
taula

## Nivell 2 · Respondre

### 2.1 · Contractes per any

Quants contractes de lloguer es van registrar cada any a tot Catalunya? Fes-ne un gràfic de línia. Explica en una línia quines files has fet servir i per què.

In [ ]:
anual = df[df["periode"] == "gener-desembre"]
contractes = anual.groupby("any")["habitatges"].sum()
print(contractes.to_string())
# el que surt si se suma tot sense mirar el període
print(df.groupby("any")["habitatges"].sum().loc[2019:2025].to_string())
contractes.plot(marker="o", title="Contractes de lloguer registrats per any (Catalunya)")
plt.show()

**Comentari.** El parany del repte: `periode` barreja l'any sencer, trimestres i acumulats (gener-juny, gener-setembre). Des del 2019, sumar-ho tot compta els mateixos contractes dues o tres vegades. I el 2026 només té el primer trimestre.

### 2.2 · On és més car

Quins són els 10 municipis amb el lloguer mitjà més car de l'**últim any complet**? Mostra també quants contractes té cada un. Te'n refiaries de tots?

In [ ]:
a25 = anual[anual["any"] == 2025]
print(a25.nlargest(10, "renda")[["nom_territori", "habitatges", "renda"]].round(0))
# amb un mínim de contractes perquè la mitjana sigui fiable
a25[a25["habitatges"] >= 100].nlargest(10, "renda")[["nom_territori", "habitatges", "renda"]].round(0)

### 2.3 · Tres ciutats en el temps

Fes una taula amb els anys a les files i **Barcelona**, **Sant Cugat del Vallès** i **Terrassa** a les columnes, amb la renda mitjana anual. Dibuixa-la.

In [ ]:
tres = ["Barcelona", "Sant Cugat del Vallès", "Terrassa"]
evol = anual[anual["nom_territori"].isin(tres)].pivot_table(index="any", columns="nom_territori", values="renda")
print(evol.shape, evol.isna().sum().sum())
print(evol.round(0).tail(12))
evol.plot(marker="o", title="Lloguer mitjà mensual (€)")
plt.show()

### 2.4 · Quant ha pujat

Calcula la pujada percentual del lloguer entre el 2015 i el 2025 a les tres ciutats. Després fes-ho per a tots els municipis que tinguin renda els dos anys i almenys 100 contractes el 2025: quins cinc han pujat més?

In [ ]:
print((100 * (evol.loc[2025] - evol.loc[2015]) / evol.loc[2015]).round(1))
t = anual.pivot_table(index="nom_territori", columns="any", values="renda")
n25 = a25.set_index("nom_territori")["habitatges"]
puja = (100 * (t[2025] - t[2015]) / t[2015]).dropna()
puja = puja[puja.index.isin(n25[n25 >= 100].index)]
print(len(puja))
puja.sort_values(ascending=False).head(5).round(1)

### 2.5 · Quin municipi s'assembla a Barcelona (NumPy)

Cada municipi té una trajectòria: la renda anual del 2015 al 2025, que són 11 números.

1. Fes un array `R` amb una fila per municipi i una columna per any (2015-2025), només amb els
   municipis que no tenen cap any buit.
2. Calcula la distància euclidiana de cada municipi a Barcelona. Quins cinc s'hi assemblen més?
3. Ara divideix cada fila pel seu valor del 2015, de manera que totes comencin a 1, i torna-ho a fer.
   Surten els mateixos? Què compara cada versió?

In [ ]:
t11 = t.loc[:, 2015:2025].dropna()
R = t11.to_numpy()
i_bcn = list(t11.index).index("Barcelona")
print(R.shape)
d = np.linalg.norm(R - R[i_bcn], axis=1)
print("a si mateixa:", d[i_bcn])
print(pd.Series(d, index=t11.index).nsmallest(6).round(0))
Rn = R / R[:, [0]]
print(np.unique(Rn[:, 0]))
dn = np.linalg.norm(Rn - Rn[i_bcn], axis=1)
pd.Series(dn, index=t11.index).nsmallest(6).round(3)

**Comentari.** Sense dividir es comparen **nivells de preu** (els veïns de Barcelona són els municipis cars); dividint pel 2015 es compara la **forma de la pujada**, i surten municipis de tota mena.

## Nivell 3 · Més enllà

### 3.1 · El 2020

Amb els trimestres (els períodes de tres mesos), mira què va passar amb el nombre de contractes i amb la renda a Barcelona el 2019, el 2020 i el 2021.

In [ ]:
trim = ["gener-març", "abril-juny", "juliol-setembre", "octubre-desembre"]
bcn = df[(df["nom_territori"] == "Barcelona") & df["periode"].isin(trim) & df["any"].between(2019, 2021)]
print(bcn.pivot_table(index="any", columns="periode", values="habitatges")[trim])
bcn.pivot_table(index="any", columns="periode", values="renda")[trim].round(0)

### 3.2 · Els trams de preu

Què indica `tram_preus`? Té els mateixos valors tots els anys? Ho faries servir per comparar anys?

In [ ]:
print(df.groupby("any")["tram_preus"].unique().loc[[2007, 2015, 2025]].to_string())

## Conclusió de referència

Entre el 2015 i el 2025 el lloguer mitjà ha pujat un 54,4 % a Barcelona (de 735 a 1.135 €), un
59,0 % a Sant Cugat del Vallès (de 898 a 1.427 €) i un 69,4 % a Terrassa (de 446 a 756 €): Terrassa
és la més barata, però la que ha pujat més en percentatge. El 2025, el municipi amb la renda més
alta és Sant Vicenç de Montalt (1.637 €, però amb només 79 contractes); amb un mínim de 100
contractes, el més car és Sant Just Desvern (1.478 €), seguit de Sant Cugat. Els contractes anuals
van tocar sostre el 2018 (167.953) i el 2021 (167.842), i el 2025 en van ser 108.057. Parany: si se
sumen tots els períodes, el 2025 surten 351.404 contractes, perquè `periode` barreja l'any sencer,
trimestres i acumulats.